# DAC 2026 V8 Robust Boundary

This notebook preserves the V3 ranking and tests only leakage-safe hard-negative boundary rerankers. Full mode is competition-compatible on Kaggle. Fast mode is an end-to-end behavior smoke test and is non-comparable.

In [ ]:
import gc
import io
import json
import math
import os
import platform
import random
import subprocess
import time
import warnings
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from catboost import CatBoostClassifier
    from catboost.utils import get_gpu_device_count
except ImportError as error:
    raise RuntimeError('CatBoost is required for V8') from error

try:
    from xgboost import DMatrix, XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    DMatrix = XGBClassifier = None
    XGBOOST_AVAILABLE = False

try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    lgb = LGBMClassifier = None
    LIGHTGBM_AVAILABLE = False

warnings.filterwarnings('ignore', category=FutureWarning)
if not LIGHTGBM_AVAILABLE and not (os.getenv('DAC_FAST_MODE', '0') == '1'):
    raise RuntimeError('LightGBM unavailable in full mode. Install the declared notebook dependency before training.')

@dataclass(frozen=True)
class Config:
    seed: int = 2026
    folds: int = 5
    stress_folds: int = 5
    inner_folds: int = 3
    fast_mode: bool = os.getenv('DAC_FAST_MODE', '0') == '1'
    cat_iterations: int = 2860
    xgb_estimators: int = 2600
    lgbm_estimators: int = 2600
    logistic_max_iter: int = 700
    refiner_iterations: int = 900
    hn_iterations: int = 700
    candidate_pool_rate: float = 0.10
    hn_legitimate_weight: float = 12.0
    te_alpha: float = 50.0
    hn_te_alpha: float = 12.0
    action_bands: tuple = ((0.040, 0.065), (0.035, 0.070))
    surgical_weights: tuple = (0.35, 0.60, 0.85)
    specialist_full_seeds: tuple = (2026, 2127, 3407)
    min_nr5_gain: float = 0.001
    min_fold_wins: int = 3
    min_confirmation_gain: int = 0
    max_worst_fold_loss: float = -2.0
    min_ap_delta: float = -0.001
    max_oof_churn: float = 0.08
    min_gain_per_100_swaps: float = 1.0
    max_test_churn: float = 0.10
    min_nonfraud_group: int = 100

CFG = Config()
if CFG.fast_mode:
    CFG = Config(
        fast_mode=True, folds=2, stress_folds=2, inner_folds=2,
        cat_iterations=180, xgb_estimators=180, lgbm_estimators=180,
        logistic_max_iter=180, refiner_iterations=120, hn_iterations=120,
        specialist_full_seeds=(2026,),
    )

IS_KAGGLE = Path('/kaggle/input').exists()
OUTPUT_DIR = Path('/kaggle/working/v8_artifacts' if IS_KAGGLE else '/tmp/dac2026_v8_artifacts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = Path('/kaggle/working/submission_v8.csv' if IS_KAGGLE else '/tmp/submission_v8.csv')
random.seed(CFG.seed)
np.random.seed(CFG.seed)
os.environ['PYTHONHASHSEED'] = str(CFG.seed)

GPU_COUNT = int(get_gpu_device_count())
CAT_TASK = 'GPU' if GPU_COUNT else 'CPU'
CAT_DEVICES = ':'.join(str(i) for i in range(GPU_COUNT)) if GPU_COUNT else None
HARDWARE = 'CPU runtime'
if GPU_COUNT:
    HARDWARE = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True, check=False,
    ).stdout.strip().replace('\n', '; ')

RUNTIME_LABEL = 'FAST smoke, non-comparable' if CFG.fast_mode else 'FULL competition'
print(json.dumps({'mode': RUNTIME_LABEL, 'python': platform.python_version(), 'gpus': GPU_COUNT,
                  'catboost_devices': CAT_DEVICES or 'CPU', 'lightgbm_available': LIGHTGBM_AVAILABLE,
                  'hardware': HARDWARE}, indent=2))


## 1. Metrics and neutral top-k selection

The selector uses claim ID only as a deterministic tie key. It is never passed to a model.

In [ ]:
ID_COL = 'claim_id'
TARGET_COL = 'label'

def _binary(values):
    values = np.asarray(values)
    if values.ndim != 1 or not np.isin(values, [0, 1]).all():
        raise ValueError('labels must be one-dimensional binary values')
    return values.astype(int)

def _scores(values):
    values = np.asarray(values, dtype=float)
    if values.ndim != 1 or not np.isfinite(values).all():
        raise ValueError('scores must be finite one-dimensional values')
    return values

def _probabilities(values):
    values = _scores(values)
    if ((values < 0) | (values > 1)).any():
        raise ValueError('probabilities must be in [0, 1]')
    return values

def top_k_mask(scores, capacity=0.05, tie_key=None):
    score = _scores(scores)
    if not 0 <= capacity <= 1:
        raise ValueError('capacity must be in [0, 1]')
    k = int(np.floor(capacity * len(score)))
    out = np.zeros(len(score), dtype=bool)
    if k == 0:
        return out
    if tie_key is None:
        order = np.argsort(-score, kind='mergesort')
        if k < len(score) and score[order[k - 1]] == score[order[k]]:
            raise ValueError('label-neutral tie_key required when a tie crosses boundary')
    else:
        keys = pd.Series(tie_key, dtype='string').fillna('__MISSING_KEY__').astype(str).to_numpy()
        if len(keys) != len(score):
            raise ValueError('tie_key length mismatch')
        order = np.lexsort((keys, -score))
    out[order[:k]] = True
    return out

def normalized_recall_at_capacity(y_true, score, capacity=0.05, tie_key=None):
    y = _binary(y_true)
    selected = top_k_mask(score, capacity, tie_key)
    positives = int(y.sum())
    if positives == 0 or not selected.any():
        return np.nan
    recall = y[selected].sum() / positives
    return float(recall / min(1.0, selected.sum() / positives))

def age_groups(age):
    values = pd.to_numeric(pd.Series(age), errors='coerce').mask(lambda s: s < 0)
    bands = pd.cut(values, [-np.inf, 17, 39, 59, np.inf], labels=['0-17', '18-39', '40-59', '60+'])
    return pd.Series(bands, index=values.index).astype('object').fillna('Missing')

def protection_score(y_true, selected, gender, age):
    y = _binary(y_true)
    selected = np.asarray(selected, dtype=bool)
    def gap(values, levels=None):
        s = pd.Series(values, dtype='string').fillna('Missing').str.strip().replace('', 'Missing')
        levels = levels or sorted(s.unique())
        fprs = []
        for level in levels:
            member = s.eq(level).to_numpy()
            legitimate = member & (y == 0)
            if legitimate.sum() >= CFG.min_nonfraud_group and str(level) != 'Missing':
                fprs.append(float((legitimate & selected).sum() / legitimate.sum()))
        return max(fprs) - min(fprs) if len(fprs) >= 2 else 0.0
    gg, ag = gap(gender), gap(age_groups(age), ['0-17', '18-39', '40-59', '60+'])
    return 1.0 - max(gg, ag), gg, ag

def metric_row(y_true, score, tie_key, gender, age, name, probability=None):
    y = _binary(y_true); score = _scores(score)
    selected = top_k_mask(score, 0.05, tie_key)
    protection, gg, ag = protection_score(y, selected, gender, age)
    row = {'candidate': name, 'NormalizedRecall@5%': normalized_recall_at_capacity(y, score, 0.05, tie_key),
           'AveragePrecision': float(average_precision_score(y, score)), 'fraud_captured': int(y[selected].sum()),
           'legitimate_audits': int(((y == 0) & selected).sum()), 'ProtectionScore': protection,
           'Gap_gender': gg, 'Gap_age': ag}
    if probability is not None:
        p = _probabilities(probability)
        row['Brier'] = float(brier_score_loss(y, p)); row['CalibrationScore'] = 1.0 - row['Brier']
    return row

def fold_metrics(y_true, score, fold_id, tie_key):
    rows = []
    for fold in sorted(np.unique(fold_id)):
        idx = np.flatnonzero(fold_id == fold)
        rows.append({'fold': int(fold) + 1, 'NR5': normalized_recall_at_capacity(np.asarray(y_true)[idx], np.asarray(score)[idx], 0.05, np.asarray(tie_key)[idx]),
                     'tp': int(_binary(y_true)[idx][top_k_mask(np.asarray(score)[idx], 0.05, np.asarray(tie_key)[idx])].sum())})
    return pd.DataFrame(rows)

_t_y = np.array([1, 0, 1, 0, 1, 0, 1, 0]); _t_s = np.array([0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2])
assert top_k_mask(_t_s, 0.25, [f'c{i}' for i in range(8)]).sum() == 2
assert 0 <= normalized_recall_at_capacity(_t_y, _t_s, 0.25, [f'c{i}' for i in range(8)]) <= 1
try:
    top_k_mask(np.array([1.0, 0.2, 0.2]), 2 / 3)
except ValueError:
    pass
else:
    raise AssertionError('boundary ties require a label-neutral tie key')
print('Behavior 1 metric-neutral top-k boundary cases: PASS')


## 2. Data discovery and deterministic feature package

In [ ]:
SEMANTIC_CATEGORICAL = {'kdkc', 'dati2', 'typeppk', 'jkpst', 'jnspelsep', 'cmg', 'severitylevel', 'diagprimer'}
FORBIDDEN_INPUT_TOKENS = ('ground_truth',)

def forbidden_path(path):
    return any(token in str(path).lower() for token in FORBIDDEN_INPUT_TOKENS)

def csv_schema(path):
    return pd.read_csv(path, nrows=0).columns.tolist()

def discover_competition_files():
    roots = [Path('/kaggle/input')] if IS_KAGGLE else [Path.cwd() / 'data', Path.cwd().parent / 'data', Path.cwd()]
    candidates = {'train.csv': [], 'test.csv': [], 'sample_submission.csv': []}
    for root in roots:
        if root.exists():
            for name in candidates:
                candidates[name].extend([p for p in root.rglob(name) if not forbidden_path(p)])
    candidates = {name: sorted(set(paths), key=lambda p: p.stat().st_size, reverse=True) for name, paths in candidates.items()}
    for train_path in candidates['train.csv']:
        train_cols = csv_schema(train_path)
        if ID_COL not in train_cols or TARGET_COL not in train_cols:
            continue
        for test_path in candidates['test.csv']:
            if set(csv_schema(test_path)) != set(train_cols) - {TARGET_COL}:
                continue
            test_ids = pd.read_csv(test_path, usecols=[ID_COL], dtype={ID_COL: 'string'})[ID_COL]
            for sample_path in candidates['sample_submission.csv']:
                if csv_schema(sample_path) != [ID_COL, 'fraud_probability']:
                    continue
                sample_ids = pd.read_csv(sample_path, usecols=[ID_COL], dtype={ID_COL: 'string'})[ID_COL]
                if len(test_ids) == len(sample_ids) and set(test_ids) == set(sample_ids):
                    return train_path, test_path, sample_path
    raise FileNotFoundError('No schema-consistent DAC train/test/sample trio found')

def read_claim_csv(path):
    if forbidden_path(path):
        raise ValueError(f'forbidden helper path: {path}')
    columns = csv_schema(path)
    strings = [c for c in ({ID_COL} | SEMANTIC_CATEGORICAL) if c in columns]
    return pd.read_csv(path, dtype={c: 'string' for c in strings}, low_memory=False)

TRAIN_PATH, TEST_PATH, SAMPLE_PATH = discover_competition_files()
train_raw, test_raw, sample_submission = map(read_claim_csv, [TRAIN_PATH, TEST_PATH, SAMPLE_PATH])
assert ID_COL in train_raw and TARGET_COL in train_raw and TARGET_COL not in test_raw
assert not train_raw[ID_COL].duplicated().any() and not test_raw[ID_COL].duplicated().any()
assert list(sample_submission.columns) == [ID_COL, 'fraud_probability']
assert len(sample_submission) == len(test_raw) and set(sample_submission[ID_COL]) == set(test_raw[ID_COL])
test_raw = (sample_submission[[ID_COL]].assign(_order=np.arange(len(sample_submission)))
            .merge(test_raw, on=ID_COL, how='left', validate='one_to_one', indicator=True)
            .sort_values('_order').reset_index(drop=True))
assert test_raw['_merge'].eq('both').all()
test_raw = test_raw.drop(columns=['_order', '_merge'])

def clean_claims(frame):
    out = frame.copy()
    for column in out.columns:
        if column == TARGET_COL:
            out[column] = pd.to_numeric(out[column], errors='raise')
        elif column == ID_COL or column in SEMANTIC_CATEGORICAL or str(out[column].dtype) in {'object', 'string'}:
            values = out[column].astype('string').str.strip()
            out[column] = values.mask(values.eq(''), '__MISSING__').fillna('__MISSING__')
        else:
            out[column] = pd.to_numeric(out[column], errors='coerce')
    return out

def entropy_and_share(block):
    values = block.to_numpy(dtype=float); totals = values.sum(axis=1)
    shares = np.divide(values, totals[:, None], out=np.zeros_like(values), where=totals[:, None] > 0)
    return (-(np.where(shares > 0, shares * np.log(shares + 1e-12), 0)).sum(axis=1),
            np.divide(values.max(axis=1), totals, out=np.zeros(len(values)), where=totals > 0))

def engineer_features(frame):
    out = frame.copy(); dx = [c for c in out if c.startswith('dx2_')]; proc = [c for c in out if c.startswith('proc')]
    age = pd.to_numeric(out.get('umur'), errors='coerce'); los = pd.to_numeric(out.get('los'), errors='coerce').clip(lower=0)
    out['age_band'] = age_groups(age).astype('string'); out['is_pediatric'] = age.le(17).fillna(False).astype(int)
    out['is_older_adult'] = age.ge(60).fillna(False).astype(int); out['zero_los'] = los.eq(0).fillna(False).astype(int)
    out['log1p_los'] = np.log1p(los); out['age_x_log_los'] = age * out['log1p_los']
    for prefix, columns in [('diagnosis', dx), ('procedure', proc)]:
        block = out[columns].apply(pd.to_numeric, errors='coerce').fillna(0).clip(lower=0)
        out[f'{prefix}_total'] = block.sum(axis=1); out[f'{prefix}_nonzero_groups'] = block.ne(0).sum(axis=1)
        out[f'{prefix}_max_group'] = block.max(axis=1); entropy, share = entropy_and_share(block)
        out[f'{prefix}_entropy'] = entropy; out[f'{prefix}_max_share'] = share
    out['total_coded_burden'] = out['diagnosis_total'] + out['procedure_total']
    out['total_nonzero_groups'] = out['diagnosis_nonzero_groups'] + out['procedure_nonzero_groups']
    out['log1p_coded_burden'] = np.log1p(out['total_coded_burden'])
    out['procedure_to_diagnosis'] = (out['procedure_total'] + 0.5) / (out['diagnosis_total'] + 0.5)
    out['burden_per_day'] = out['total_coded_burden'] / (los.fillna(0) + 1)
    out['procedure_per_day'] = out['procedure_total'] / (los.fillna(0) + 1)
    out['diagnosis_per_day'] = out['diagnosis_total'] / (los.fillna(0) + 1)
    for source in ['kdkc', 'dati2']:
        if source in out: out[f'{source}_prefix2'] = out[source].astype('string').str[:2]
    interactions = {'cmg__severity': ('cmg', 'severitylevel'), 'service__facility': ('jnspelsep', 'typeppk'),
                    'geo__facility': ('dati2', 'typeppk'), 'kdkc__dati2': ('kdkc', 'dati2'),
                    'diagnosis__cmg': ('diagprimer', 'cmg'), 'diagnosis__service': ('diagprimer', 'jnspelsep'),
                    'diagnosis__facility': ('diagprimer', 'typeppk'), 'cmg__facility': ('cmg', 'typeppk')}
    for name, (left, right) in interactions.items():
        if left in out and right in out: out[name] = out[left].astype('string') + '__' + out[right].astype('string')
    return out

train_clean, test_clean = clean_claims(train_raw), clean_claims(test_raw)
train_model, test_model = engineer_features(train_clean), engineer_features(test_clean)
y = pd.to_numeric(train_model[TARGET_COL], errors='raise').astype(int).to_numpy()
assert np.isin(y, [0, 1]).all()
TIE_KEYS = train_raw[ID_COL].astype('string').astype(str).to_numpy(); TEST_TIE_KEYS = test_raw[ID_COL].astype('string').astype(str).to_numpy()
CONSTANT_COLUMNS = [c for c in train_model if c not in {ID_COL, TARGET_COL} and train_model[c].nunique(dropna=False) <= 1]
FULL_FEATURES = [c for c in train_model if c not in {ID_COL, TARGET_COL, *CONSTANT_COLUMNS}]
full_forbidden = ('kdkc', 'dati2', 'typeppk', 'geo__facility', 'service__facility', 'kdkc__dati2', 'diagnosis__facility', 'cmg__facility', 'frequency__')
PROXY_LIGHT_FEATURES = [c for c in FULL_FEATURES if not any(token in c.lower() for token in full_forbidden)]
assert ID_COL not in FULL_FEATURES and TARGET_COL not in FULL_FEATURES
assert not any(any(token in c.lower() for token in full_forbidden) for c in PROXY_LIGHT_FEATURES)
CATEGORICAL_COLUMNS = [c for c in FULL_FEATURES if c in SEMANTIC_CATEGORICAL or str(train_model[c].dtype) in {'object', 'string'}]
print({'train_rows': len(train_model), 'test_rows': len(test_model), 'features': len(FULL_FEATURES),
       'proxy_light_features': len(PROXY_LIGHT_FEATURES), 'fraud_rate': float(y.mean())})
print('Behavior 5 proxy-light forbidden-column assertion: PASS')


## 3. Grouped validation and fold-safe encodings

In [ ]:
signature_frame = train_clean.drop(columns=[ID_COL, TARGET_COL], errors='ignore')
SIGNATURE_GROUPS = pd.util.hash_pandas_object(signature_frame, index=False).astype('uint64').astype('string')

def make_splits(groups, n_splits, seed, stratify=y):
    if pd.Series(groups).nunique() < n_splits:
        raise ValueError('not enough groups for requested grouped folds')
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(splitter.split(np.zeros(len(y)), stratify, groups=groups))
PRIMARY_SPLITS = make_splits(SIGNATURE_GROUPS, CFG.folds, CFG.seed)
FOLD_ID = np.full(len(y), -1, dtype=int)
fold_rows = []
for fold, (fit_idx, valid_idx) in enumerate(PRIMARY_SPLITS):
    FOLD_ID[valid_idx] = fold
    overlap = set(SIGNATURE_GROUPS.iloc[fit_idx]) & set(SIGNATURE_GROUPS.iloc[valid_idx])
    fold_rows.append({'fold': fold + 1, 'train_n': len(fit_idx), 'valid_n': len(valid_idx),
                      'valid_fraud_rate': float(y[valid_idx].mean()), 'signature_overlap': len(overlap),
                      'role': 'selection' if fold < min(3, CFG.folds) else 'confirmation'})
VALIDATION_TABLE = pd.DataFrame(fold_rows)
assert (FOLD_ID >= 0).all() and VALIDATION_TABLE.signature_overlap.sum() == 0
VALIDATION_TABLE.to_csv(OUTPUT_DIR / 'primary_fold_assignments_summary.csv', index=False)
pd.DataFrame({ID_COL: TIE_KEYS, 'fold': FOLD_ID}).to_csv(OUTPUT_DIR / 'primary_fold_assignments.csv', index=False)
print(VALIDATION_TABLE.to_string(index=False)); print('Behavior 2 exact-signature disjointness: PASS')

TE_SPECS = {'dati2_typeppk_cmg': ('dati2', 'typeppk', 'cmg'), 'dati2_diagprimer_cmg': ('dati2', 'diagprimer', 'cmg'),
            'kdkc_typeppk_cmg': ('kdkc', 'typeppk', 'cmg'), 'dati2_typeppk_diagprimer': ('dati2', 'typeppk', 'diagprimer'),
            'diagprimer_cmg_typeppk': ('diagprimer', 'cmg', 'typeppk'), 'dati2_jnspelsep_typeppk': ('dati2', 'jnspelsep', 'typeppk')}
TE_SPECS = {name: tuple(c for c in cols if c in train_model) for name, cols in TE_SPECS.items()}
TE_SPECS = {name: cols for name, cols in TE_SPECS.items() if len(cols) >= 2}

def target_encoding_map(fit_frame, fit_target, apply_frame, columns, alpha):
    fit_keys = pd.MultiIndex.from_frame(fit_frame[list(columns)].astype('string'))
    apply_keys = pd.MultiIndex.from_frame(apply_frame[list(columns)].astype('string'))
    stats = pd.DataFrame({'_target': np.asarray(fit_target, dtype=float)}, index=fit_keys).groupby(level=list(range(len(columns))))['_target'].agg(['sum', 'count'])
    prior = float(np.mean(fit_target)); rate = (stats['sum'] + alpha * prior) / (stats['count'] + alpha)
    mapped_rate = rate.reindex(apply_keys).to_numpy(dtype=float); mapped_count = stats['count'].reindex(apply_keys).to_numpy(dtype=float)
    return np.where(np.isfinite(mapped_rate), mapped_rate, prior), np.log1p(np.where(np.isfinite(mapped_count), mapped_count, 0.0))

def build_fold_safe_te():
    oof = pd.DataFrame(index=train_model.index); test_parts = []
    for _, (fit_idx, valid_idx) in enumerate(PRIMARY_SPLITS):
        part = pd.DataFrame(index=test_model.index)
        for name, cols in TE_SPECS.items():
            vr, vc = target_encoding_map(train_model.iloc[fit_idx], y[fit_idx], train_model.iloc[valid_idx], cols, CFG.te_alpha)
            tr, tc = target_encoding_map(train_model.iloc[fit_idx], y[fit_idx], test_model, cols, CFG.te_alpha)
            oof.loc[valid_idx, f'te__{name}'] = vr; oof.loc[valid_idx, f'te_count__{name}'] = vc
            part[f'te__{name}'] = tr; part[f'te_count__{name}'] = tc
        test_parts.append(part)
    if oof.isna().any().any(): raise RuntimeError('incomplete outer target encodings')
    return oof.astype(float), (sum(test_parts) / len(test_parts)).astype(float)
TE_OOF, TE_TEST = build_fold_safe_te()
np.savez_compressed(OUTPUT_DIR / 'hierarchical_target_estimates.npz', oof=TE_OOF.to_numpy(np.float32), test=TE_TEST.to_numpy(np.float32), columns=np.array(TE_OOF.columns, dtype=object))
assert np.isfinite(TE_OOF.to_numpy()).all() and np.isfinite(TE_TEST.to_numpy()).all()
print('Behavior 6 outer target-encoding completeness: PASS')


## 4. Artifact-first base layer

In [ ]:
EXPECTED_ARTIFACTS = {'base_fold_metrics.csv', 'base_model_summary.csv', 'hierarchical_target_estimates.npz', 'meta_features.npz',
                      'primary_fold_assignments.csv', 'primary_fold_assignments_summary.csv', 'stack_comparison.csv', 'stack_fold_comparison.csv',
                      'predictions_catboost_v3_seed_2026.npz', 'predictions_catboost_v3_seed_2127.npz', 'predictions_xgboost_v3_one-hot.npz',
                      'predictions_logistic_v3.npz', 'predictions_lightgbm_v3_one-hot.npz'}

def artifact_candidates():
    paths = []
    roots = [Path('/kaggle/input'), Path.cwd(), Path.cwd().parent, Path('/home/athilla/Documents/IF_ITB/Lomba/DAC-PRS-ITS/prs-its')]
    for root in roots:
        if root.exists():
            paths.extend(root.rglob('results.zip'))
        paths.extend([p for p in root.rglob('v7_artifacts') if p.is_dir()])
        paths.extend([p for p in root.rglob('v8_artifacts') if p.is_dir()])
    return sorted(set(paths), key=lambda p: (p.is_dir(), -p.stat().st_size if p.is_file() else 0))

def open_artifact_store(path):
    if path.is_dir():
        return {'names': {p.name for p in path.iterdir()}, 'read': lambda name: path.joinpath(name).read_bytes()}
    archive = zipfile.ZipFile(path)
    names = {Path(n).name for n in archive.namelist() if n.endswith('/') is False}
    return {'names': names, 'read': lambda name: archive.read(next(n for n in archive.namelist() if n.endswith('/' + name) or n == name))}

def validate_artifacts(store):
    missing = EXPECTED_ARTIFACTS - store['names']
    if missing: raise RuntimeError(f'present V7 artifacts are invalid, missing: {sorted(missing)}')
    summary = pd.read_csv(io.BytesIO(store['read']('base_model_summary.csv')))
    assignments = pd.read_csv(io.BytesIO(store['read']('primary_fold_assignments.csv')))
    if len(assignments) != len(train_model) or assignments[ID_COL].astype(str).tolist() != TIE_KEYS.tolist():
        raise RuntimeError('V7 fold assignment shape or ID order invalid')
    artifact_folds = assignments.fold.to_numpy(dtype=int)
    artifact_levels = sorted(np.unique(artifact_folds).tolist())
    if artifact_levels == [1, 2, 3, 4, 5]: artifact_folds = artifact_folds - 1
    elif artifact_levels != [0, 1, 2, 3, 4]: raise RuntimeError('V7 fold coverage invalid')
    expected_names = {'CatBoost V3 seed 2026', 'CatBoost V3 seed 2127', 'XGBoost V3 one-hot', 'Logistic V3', 'LightGBM V3 one-hot'}
    if set(summary.candidate) != expected_names: raise RuntimeError('V7 model-name contract invalid')
    arrays = {}
    for name in expected_names:
        safe = name.lower().replace(' ', '_').replace('/', '_')
        data = np.load(io.BytesIO(store['read'](f'predictions_{safe}.npz')), allow_pickle=True)
        if data['oof'].shape != (len(train_model),) or data['test'].shape != (len(test_model),): raise RuntimeError(f'invalid shape for {name}')
        if not np.isfinite(data['oof']).all() or not np.isfinite(data['test']).all(): raise RuntimeError(f'nonfinite predictions for {name}')
        arrays[name] = {'name': name, 'kind': name.split()[0], 'oof': np.clip(data['oof'], 1e-7, 1 - 1e-7), 'test': np.clip(data['test'], 1e-7, 1 - 1e-7)}
    meta = np.load(io.BytesIO(store['read']('meta_features.npz')), allow_pickle=True)
    meta_columns = np.asarray(meta['columns']).astype(str)
    if (meta['oof'].shape[0] != len(train_model) or meta['test'].shape[0] != len(test_model) or
            meta['oof'].shape[1] != meta['test'].shape[1] or len(meta_columns) != meta['oof'].shape[1] or
            len(set(meta_columns.tolist())) != len(meta_columns) or not all(meta_columns)):
        raise RuntimeError('meta shape or feature-column metadata invalid')
    if not np.isfinite(meta['oof']).all() or not np.isfinite(meta['test']).all(): raise RuntimeError('meta values invalid')
    hierarchy = np.load(io.BytesIO(store['read']('hierarchical_target_estimates.npz')), allow_pickle=True)
    hierarchy_columns = np.asarray(hierarchy['columns']).astype(str) if 'columns' in hierarchy.files else np.array([], dtype=str)
    if (hierarchy['oof'].shape[0] != len(train_model) or hierarchy['test'].shape[0] != len(test_model) or
            hierarchy['oof'].shape[1] != hierarchy['test'].shape[1] or len(hierarchy_columns) != hierarchy['oof'].shape[1] or
            len(set(hierarchy_columns.tolist())) != len(hierarchy_columns) or not all(hierarchy_columns)):
        raise RuntimeError('hierarchy shape or feature-column metadata invalid')
    if not np.isfinite(hierarchy['oof']).all() or not np.isfinite(hierarchy['test']).all(): raise RuntimeError('hierarchy values invalid')
    for archive_name, data, expected_n in [('meta_features.npz', meta, len(train_model)), ('hierarchical_target_estimates.npz', hierarchy, len(train_model))]:
        for row_key in ('claim_id', 'ids', 'row_order'):
            if row_key in data.files and len(data[row_key]) != expected_n: raise RuntimeError(f'{archive_name} row-order metadata invalid')
    return arrays, pd.DataFrame(meta['oof'], columns=meta['columns'].astype(str)), pd.DataFrame(meta['test'], columns=meta['columns'].astype(str)), artifact_folds

V8_EVAL_FILES = {'v8_candidate_registry.csv', 'v8_primary_fold_deltas.csv', 'v8_stress_metrics.csv', 'v8_test_churn.csv'}
BASE_ROSTER_FALLBACK = ['CatBoost V3 seed 2127', 'CatBoost V3 seed 2026']

def expected_candidate_names():
    names = ['V3 control', 'Stack control']
    for view in ('HN full', 'HN proxy-light'):
        for band_idx, _band in enumerate(CFG.action_bands, 1):
            for weight in CFG.surgical_weights:
                names.append(f'{view} band{band_idx} w{weight:.2f}')
    return names

def load_npz_prediction(store, name):
    safe = name.lower().replace(' ', '_').replace('/', '_')
    data = np.load(io.BytesIO(store['read'](f'predictions_{safe}.npz')), allow_pickle=True)
    if data['oof'].shape != (len(train_model),) or data['test'].shape != (len(test_model),): raise RuntimeError(f'invalid shape for {name}')
    if not np.isfinite(data['oof']).all() or not np.isfinite(data['test']).all(): raise RuntimeError(f'nonfinite predictions for {name}')
    return {'name': name, 'kind': name.split()[0], 'oof': np.clip(data['oof'], 1e-7, 1 - 1e-7), 'test': np.clip(data['test'], 1e-7, 1 - 1e-7)}

def load_fold_assignments(store):
    assignments = pd.read_csv(io.BytesIO(store['read']('primary_fold_assignments.csv')))
    if len(assignments) != len(train_model) or assignments[ID_COL].astype(str).tolist() != TIE_KEYS.tolist():
        raise RuntimeError('fold assignment shape or ID order invalid')
    folds = assignments.fold.to_numpy(dtype=int)
    levels = sorted(np.unique(folds).tolist())
    if levels == [1, 2, 3, 4, 5]: folds = folds - 1
    elif levels != [0, 1, 2, 3, 4]: raise RuntimeError('fold coverage invalid')
    return folds

def validate_npz_metadata(store, key):
    data = np.load(io.BytesIO(store['read'](key)), allow_pickle=True)
    columns = np.asarray(data['columns']).astype(str) if 'columns' in data.files else np.array([], dtype=str)
    if (data['oof'].shape[0] != len(train_model) or data['test'].shape[0] != len(test_model) or
            data['oof'].shape[1] != data['test'].shape[1] or len(columns) != data['oof'].shape[1] or
            len(set(columns.tolist())) != len(columns) or not all(columns)):
        raise RuntimeError(f'{key} shape or feature-column metadata invalid')
    if not np.isfinite(data['oof']).all() or not np.isfinite(data['test']).all(): raise RuntimeError(f'{key} values invalid')

def validate_v8_checkpoint(store):
    missing = V8_EVAL_FILES - store['names']
    if missing: raise RuntimeError(f'V8 checkpoint evaluation files missing: {sorted(missing)}')
    registry = pd.read_csv(io.BytesIO(store['read']('v8_candidate_registry.csv')))
    required = {'candidate', 'family', 'NormalizedRecall@5%', 'AveragePrecision', 'fraud_captured', 'min_stress_delta',
                'confirmation_tp_gain', 'tp_gain_per_100_swaps', 'test_churn', *[f'gate_{i}' for i in range(1, 11)]}
    if len(registry) != 14 or set(registry.candidate) != set(expected_candidate_names()):
        raise RuntimeError('V8 checkpoint registry cardinality or candidate names invalid')
    if not required.issubset(registry.columns): raise RuntimeError('V8 checkpoint registry columns incomplete')
    if not all(registry[f'gate_{i}'].dtype == bool for i in range(1, 11)): raise RuntimeError('V8 checkpoint gate columns are not boolean')
    fold_deltas = pd.read_csv(io.BytesIO(store['read']('v8_primary_fold_deltas.csv')))
    if len(fold_deltas) != 70 or set(fold_deltas.candidate) != set(registry.candidate) or set(fold_deltas.fold.astype(int)) != {1, 2, 3, 4, 5}:
        raise RuntimeError('V8 checkpoint fold-delta table invalid')
    stress = pd.read_csv(io.BytesIO(store['read']('v8_stress_metrics.csv')))
    churn = pd.read_csv(io.BytesIO(store['read']('v8_test_churn.csv')))
    if len(stress) != 14 or not {'kdkc_stress_delta', 'dati2_stress_delta'}.issubset(stress.columns) or set(stress.candidate) != set(registry.candidate):
        raise RuntimeError('V8 checkpoint stress table invalid')
    if len(churn) != 14 or set(churn.candidate) != set(registry.candidate): raise RuntimeError('V8 checkpoint churn table invalid')
    validate_npz_metadata(store, 'meta_features.npz')
    validate_npz_metadata(store, 'hierarchical_target_estimates.npz')
    roster = {}
    for name in BASE_ROSTER_FALLBACK:
        roster[name] = load_npz_prediction(store, name)
    artifact_folds = load_fold_assignments(store)
    return registry, fold_deltas, stress, churn, roster, artifact_folds

BASE_RESULTS = {}; ARTIFACT_PROVENANCE = 'deterministic fallback training'
EVAL_FROM_CHECKPOINT = False
CHECKPOINT_PROVENANCE = None
CKPT_REGISTRY = CKPT_FOLD_DELTAS = CKPT_STRESS_METRICS = CKPT_TEST_CHURN = None
FORCE_FULL = os.getenv('DAC_FORCE_FULL', '0') == '1'
if FORCE_FULL: print('DAC_FORCE_FULL=1: checkpoint resume is disabled; running the full pipeline.')
checkpoint_candidates, v7_candidates = [], []
for candidate in artifact_candidates():
    try:
        if candidate.resolve() == OUTPUT_DIR.resolve():
            print(f'Skipping the live output directory: {candidate}'); continue
        names = open_artifact_store(candidate)['names']
    except (zipfile.BadZipFile, OSError, RuntimeError, StopIteration) as error:
        print(f'Skipping unreadable artifact candidate {candidate}: {error}'); continue
    if not FORCE_FULL and 'v8_candidate_registry.csv' in names: checkpoint_candidates.append(candidate)
    elif len(EXPECTED_ARTIFACTS & names) >= 3: v7_candidates.append(candidate)
    else: print(f'Skipping artifact candidate without a V7 or V8 contract: {candidate}')
if not FORCE_FULL:
    for candidate in checkpoint_candidates:
        try:
            CKPT_REGISTRY, CKPT_FOLD_DELTAS, CKPT_STRESS_METRICS, CKPT_TEST_CHURN, BASE_RESULTS, ARTIFACT_FOLD_ID = validate_v8_checkpoint(open_artifact_store(candidate))
        except (RuntimeError, KeyError, StopIteration) as error:
            print(f'Skipping invalid V8 checkpoint {candidate}: {error}'); continue
        EVAL_FROM_CHECKPOINT = True; CHECKPOINT_PROVENANCE = ARTIFACT_PROVENANCE = str(candidate)
        META_OOF = pd.DataFrame({f'{name}__prob': result['oof'] for name, result in BASE_RESULTS.items()})
        META_TEST = pd.DataFrame({f'{name}__prob': result['test'] for name, result in BASE_RESULTS.items()})
        print(f'Resumed validated V8 checkpoint from {candidate}: evaluation tables and base roster loaded; specialist training will be skipped.')
        break
if not BASE_RESULTS:
    for candidate in v7_candidates:
        try:
            BASE_RESULTS, META_OOF, META_TEST, ARTIFACT_FOLD_ID = validate_artifacts(open_artifact_store(candidate))
        except (RuntimeError, KeyError, StopIteration, zipfile.BadZipFile) as error:
            raise RuntimeError(f'Invalid present V7 artifact bundle: {error}') from error
        ARTIFACT_PROVENANCE = str(candidate); print(f'Loaded validated V7 artifacts from {candidate}'); break
if BASE_RESULTS:
    ARTIFACT_SPLITS = [(np.flatnonzero(ARTIFACT_FOLD_ID != fold), np.flatnonzero(ARTIFACT_FOLD_ID == fold)) for fold in range(5)]
    ARTIFACT_FOLD_MATCHES_CURRENT = bool(np.array_equal(ARTIFACT_FOLD_ID, FOLD_ID))
    STACK_FOLD_ID, STACK_SPLITS = ARTIFACT_FOLD_ID, ARTIFACT_SPLITS
    if CFG.fast_mode:
        print('Fast smoke uses two current folds for V8 specialists and the validated five-fold artifact assignments for the artifact-backed stack; results are non-comparable.')
    else:
        print(f'Artifact-backed stack uses validated five-fold V7 assignments; exact assignment match to current V8 primary: {ARTIFACT_FOLD_MATCHES_CURRENT}.')
else:
    ARTIFACT_FOLD_ID = None
    ARTIFACT_FOLD_MATCHES_CURRENT = None
    STACK_FOLD_ID, STACK_SPLITS = FOLD_ID, PRIMARY_SPLITS
    print('No V7 artifact bundle found. Running deterministic reduced CatBoost fallback roster: CatBoost V3 seed 2127, CatBoost V3 seed 2026.')

# The fallback is intentionally small and records a reduced CatBoost roster while retaining V7 feature contracts.
def catboost_ready(frame, categorical):
    out = frame.copy()
    for c in categorical:
        if c in out: out[c] = out[c].astype('string').fillna('__MISSING__').astype(str)
    return out

def fit_cat(X_fit, target, X_valid, X_test, features, seed, iterations):
    cats = [c for c in features if c in CATEGORICAL_COLUMNS]
    params = dict(loss_function='Logloss', allow_writing_files=False, verbose=False, random_seed=seed, task_type=CAT_TASK,
                  iterations=iterations, learning_rate=0.03, depth=8, l2_leaf_reg=8.0, random_strength=0.35,
                  bootstrap_type='Bayesian', bagging_temperature=0.5, max_ctr_complexity=2, border_count=128)
    if CAT_DEVICES: params['devices'] = CAT_DEVICES
    model = CatBoostClassifier(**params)
    try:
        model.fit(catboost_ready(X_fit[features], cats), target, cat_features=cats, verbose=False)
    except Exception as error:
        msg = str(error).lower()
        if CAT_TASK != 'GPU' or not any(t in msg for t in ('gpu', 'cuda', 'nccl', 'memory')): raise
        params.update(devices='0', max_ctr_complexity=1); model = CatBoostClassifier(**params)
        model.fit(catboost_ready(X_fit[features], cats), target, cat_features=cats, verbose=False)
    return model.predict_proba(catboost_ready(X_valid[features], cats))[:, 1], model.predict_proba(catboost_ready(X_test[features], cats))[:, 1]

def run_fallback_base(name, seed, kind='CatBoost'):
    oof = np.full(len(y), np.nan); tests = []
    for fold, (fit_idx, valid_idx) in enumerate(PRIMARY_SPLITS):
        if kind == 'CatBoost':
            vp, tp = fit_cat(train_model.iloc[fit_idx], y[fit_idx], train_model.iloc[valid_idx], test_model, FULL_FEATURES, seed + fold, CFG.cat_iterations)
        else:
            raise ValueError('fallback supports CatBoost only')
        oof[valid_idx] = vp; tests.append(tp)
    result = {'name': name, 'kind': kind, 'oof': np.clip(oof, 1e-7, 1-1e-7), 'test': np.clip(np.mean(tests, axis=0), 1e-7, 1-1e-7)}
    safe = name.lower().replace(' ', '_').replace('/', '_')
    np.savez_compressed(OUTPUT_DIR / f'predictions_{safe}.npz', oof=result['oof'], test=result['test'], test_folds=np.vstack(tests))
    return result

if not BASE_RESULTS:
    BASE_RESULTS['CatBoost V3 seed 2127'] = run_fallback_base('CatBoost V3 seed 2127', 2127)
    BASE_RESULTS['CatBoost V3 seed 2026'] = run_fallback_base('CatBoost V3 seed 2026', 2026)
    # Optional families are not mislabeled when fallback dependencies/artifacts are absent.
    META_OOF = pd.DataFrame({f'{name}__prob': result['oof'] for name, result in BASE_RESULTS.items()})
    META_TEST = pd.DataFrame({f'{name}__prob': result['test'] for name, result in BASE_RESULTS.items()})

# Artifact and fallback contract is checked in either path.
for result in BASE_RESULTS.values():
    assert result['oof'].shape == (len(y),) and result['test'].shape == (len(test_model),)
    assert np.isfinite(result['oof']).all() and np.isfinite(result['test']).all()
BASE_FOLD_SUMMARY = pd.DataFrame([metric_row(y, r['oof'], TIE_KEYS, train_clean['jkpst'], train_clean['umur'], n, r['oof']) for n, r in BASE_RESULTS.items()])
BASE_FOLD_SUMMARY.to_csv(OUTPUT_DIR / 'base_model_summary.csv', index=False)
BASE_FOLD_ROWS = []
for model_name, result in BASE_RESULTS.items():
    for row in fold_metrics(y, result['oof'], FOLD_ID, TIE_KEYS).to_dict('records'):
        BASE_FOLD_ROWS.append({'model': model_name, **row})
pd.DataFrame(BASE_FOLD_ROWS).to_csv(OUTPUT_DIR / 'base_fold_metrics.csv', index=False)
print('Behavior 4 artifact shape/model/fold/finite-value contracts: PASS')


## 5. Cross-fitted stack and frozen V3 control

In [ ]:
def probability_logit(values):
    p = np.clip(np.asarray(values, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def fold_local_percentile(values, fold_id):
    values = np.asarray(values, dtype=float); out = np.zeros(len(values))
    for fold in sorted(np.unique(fold_id)):
        idx = np.flatnonzero(fold_id == fold); out[idx] = pd.Series(values[idx]).rank(method='average', pct=True).to_numpy()
    return out

def global_percentile(values): return pd.Series(np.asarray(values, dtype=float)).rank(method='average', pct=True).to_numpy()

META_OOF = META_OOF.reset_index(drop=True).astype(float); META_TEST = META_TEST.reset_index(drop=True).astype(float)
for name, result in BASE_RESULTS.items():
    key = f'base__{name.lower().replace(" ", "_").replace("-", "_")}'
    META_OOF[f'{key}__logit'] = probability_logit(result['oof']); META_OOF[f'{key}__rank'] = fold_local_percentile(result['oof'], STACK_FOLD_ID)
    META_TEST[f'{key}__logit'] = probability_logit(result['test']); META_TEST[f'{key}__rank'] = global_percentile(result['test'])
prob_oof = np.column_stack([r['oof'] for r in BASE_RESULTS.values()]); prob_test = np.column_stack([r['test'] for r in BASE_RESULTS.values()])
for frame, matrix in [(META_OOF, prob_oof), (META_TEST, prob_test)]:
    frame['ensemble__mean'] = matrix.mean(axis=1); frame['ensemble__std'] = matrix.std(axis=1)
    frame['ensemble__min'] = matrix.min(axis=1); frame['ensemble__max'] = matrix.max(axis=1)
META_OOF = META_OOF.replace([np.inf, -np.inf], np.nan).fillna(0.0); META_TEST = META_TEST.replace([np.inf, -np.inf], np.nan).fillna(0.0)
np.savez_compressed(OUTPUT_DIR / 'meta_features.npz', oof=META_OOF.to_numpy(np.float32), test=META_TEST.to_numpy(np.float32), columns=np.array(META_OOF.columns, dtype=object))

def make_meta_model(seed):
    return Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(C=1.0, max_iter=CFG.logistic_max_iter, solver='lbfgs', random_state=seed))])

def crossfit_stack():
    oof = np.full(len(y), np.nan)
    for fold, (fit_idx, valid_idx) in enumerate(STACK_SPLITS):
        model = make_meta_model(CFG.seed + 900 + fold); model.fit(META_OOF.iloc[fit_idx], y[fit_idx]); oof[valid_idx] = model.predict_proba(META_OOF.iloc[valid_idx])[:, 1]
    if np.isnan(oof).any(): raise RuntimeError('stack OOF incomplete')
    final = make_meta_model(CFG.seed + 999); final.fit(META_OOF, y)
    return np.clip(oof, 1e-7, 1-1e-7), np.clip(final.predict_proba(META_TEST)[:, 1], 1e-7, 1-1e-7)
STACK_OOF, STACK_TEST = crossfit_stack(); STACK_RANK_OOF = fold_local_percentile(STACK_OOF, STACK_FOLD_ID); STACK_RANK_TEST = global_percentile(STACK_TEST)
ANCHOR_NAME = 'CatBoost V3 seed 2127' if 'CatBoost V3 seed 2127' in BASE_RESULTS else next(iter(BASE_RESULTS))
ANCHOR_OOF, ANCHOR_TEST = BASE_RESULTS[ANCHOR_NAME]['oof'], BASE_RESULTS[ANCHOR_NAME]['test']
STACK_COMPARISON = pd.DataFrame([metric_row(y, ANCHOR_OOF, TIE_KEYS, train_clean['jkpst'], train_clean['umur'], 'V3 anchor', ANCHOR_OOF),
                                 metric_row(y, STACK_OOF, TIE_KEYS, train_clean['jkpst'], train_clean['umur'], 'Cross-fitted logistic stack', STACK_OOF)])
STACK_COMPARISON.to_csv(OUTPUT_DIR / 'stack_comparison.csv', index=False)
STACK_FOLD = fold_metrics(y, STACK_OOF, FOLD_ID, TIE_KEYS).merge(fold_metrics(y, ANCHOR_OOF, FOLD_ID, TIE_KEYS), on='fold', suffixes=('_stack', '_anchor'))
STACK_FOLD['delta'] = STACK_FOLD['NR5_stack'] - STACK_FOLD['NR5_anchor']
STACK_FOLD.to_csv(OUTPUT_DIR / 'stack_fold_comparison.csv', index=False)

def candidate_mask_from_rank(rank, rate): return np.asarray(rank) >= (1.0 - rate - 1e-12)

def fit_refiner(X_fit, target, X_valid, seed):
    cats = [c for c in FULL_FEATURES if c in CATEGORICAL_COLUMNS]
    params = dict(loss_function='Logloss', allow_writing_files=False, verbose=False, random_seed=seed, task_type=CAT_TASK,
                  iterations=CFG.refiner_iterations, learning_rate=0.035, depth=7, l2_leaf_reg=10.0,
                  random_strength=0.35, bootstrap_type='Bayesian', bagging_temperature=0.5, max_ctr_complexity=2, border_count=128)
    if CAT_DEVICES: params['devices'] = CAT_DEVICES
    model = CatBoostClassifier(**params)
    try: model.fit(catboost_ready(X_fit[FULL_FEATURES], cats), target, cat_features=cats, verbose=False)
    except Exception as error:
        msg = str(error).lower()
        if CAT_TASK != 'GPU' or not any(t in msg for t in ('gpu', 'cuda', 'nccl', 'memory')): raise
        params.update(devices='0', max_ctr_complexity=1); model = CatBoostClassifier(**params)
        model.fit(catboost_ready(X_fit[FULL_FEATURES], cats), target, cat_features=cats, verbose=False)
    return model.predict_proba(catboost_ready(X_valid[FULL_FEATURES], cats))[:, 1]

def boundary_rank_score(base_rank, refiner_prob, candidate_mask, rate, weight):
    base_rank = np.asarray(base_rank, dtype=float); refiner_prob = np.asarray(refiner_prob, dtype=float); candidate_mask = np.asarray(candidate_mask, dtype=bool)
    out = base_rank.copy(); idx = np.flatnonzero(candidate_mask)
    if not len(idx): return out
    local_base = np.clip((base_rank[idx] - (1-rate)) / rate, 0, 1)
    ref_rank = pd.Series(refiner_prob[idx]).rank(method='average', pct=True).to_numpy()
    out[idx] = np.clip((1-weight) * local_base + weight * ref_rank, 0, 1) * rate + (1-rate)
    return out

def crossfit_v3():
    candidate_all = candidate_mask_from_rank(STACK_RANK_OOF, CFG.candidate_pool_rate); ref_oof = np.full(len(y), np.nan)
    for fold, (fit_idx, valid_idx) in enumerate(PRIMARY_SPLITS):
        fit_candidates = fit_idx[candidate_all[fit_idx]]; valid_candidates = valid_idx[candidate_all[valid_idx]]
        if len(np.unique(y[fit_candidates])) < 2: raise RuntimeError(f'V3 candidate pool single class in fold {fold + 1}')
        ref_oof[valid_candidates] = fit_refiner(train_model.iloc[fit_candidates], y[fit_candidates], train_model.iloc[valid_candidates], CFG.seed + 3000 + fold)
    if np.isnan(ref_oof[candidate_all]).any(): raise RuntimeError('V3 refiner OOF incomplete')
    test_candidates = candidate_mask_from_rank(STACK_RANK_TEST, CFG.candidate_pool_rate); full_candidates = np.flatnonzero(candidate_all)
    test_ref = np.full(len(test_model), np.nan); test_ref[test_candidates] = fit_refiner(train_model.iloc[full_candidates], y[full_candidates], test_model.iloc[test_candidates], CFG.seed + 3900)
    return candidate_all, ref_oof, test_candidates, test_ref
V3_CAND_OOF, V3_REF_OOF, V3_CAND_TEST, V3_REF_TEST = crossfit_v3()
V3_OOF_SCORE = boundary_rank_score(STACK_RANK_OOF, V3_REF_OOF, V3_CAND_OOF, CFG.candidate_pool_rate, 1.0)
V3_TEST_SCORE = boundary_rank_score(STACK_RANK_TEST, V3_REF_TEST, V3_CAND_TEST, CFG.candidate_pool_rate, 1.0)
V3_FOLD_RANK_OOF, V3_RANK_TEST = fold_local_percentile(V3_OOF_SCORE, FOLD_ID), global_percentile(V3_TEST_SCORE)
V3_METRIC = metric_row(y, V3_OOF_SCORE, TIE_KEYS, train_clean['jkpst'], train_clean['umur'], 'Frozen V3 boundary control')
print(pd.DataFrame([V3_METRIC]).to_string(index=False))


## 6. Full and proxy-light hard-negative specialists

In [ ]:
HARD_NEGATIVE_TARGET = (1 - y).astype(np.int8)
HN_SPECS_FULL = {'dati2_typeppk_cmg': ('dati2', 'typeppk', 'cmg'), 'dati2_diagprimer_typeppk': ('dati2', 'diagprimer', 'typeppk'),
                 'kdkc_typeppk_cmg': ('kdkc', 'typeppk', 'cmg'), 'diagprimer_cmg_typeppk': ('diagprimer', 'cmg', 'typeppk'),
                 'dati2_severity_cmg': ('dati2', 'severitylevel', 'cmg'), 'dati2_jnspelsep_typeppk': ('dati2', 'jnspelsep', 'typeppk')}
HN_SPECS_PROXY = {'diagprimer_cmg': ('diagprimer', 'cmg'), 'severity_cmg': ('severitylevel', 'cmg'), 'diagprimer_severity_service': ('diagprimer', 'severitylevel', 'jnspelsep')}
HN_SPECS_FULL = {n: tuple(c for c in cs if c in train_model) for n, cs in HN_SPECS_FULL.items()}; HN_SPECS_PROXY = {n: tuple(c for c in cs if c in train_model) for n, cs in HN_SPECS_PROXY.items()}
HN_SPECS_FULL = {n: cs for n, cs in HN_SPECS_FULL.items() if len(cs) >= 2}; HN_SPECS_PROXY = {n: cs for n, cs in HN_SPECS_PROXY.items() if len(cs) >= 2}

def hn_encoding_block(fit_idx, apply_frame, specs):
    out = pd.DataFrame(index=np.arange(len(apply_frame)))
    for name, cols in specs.items():
        rate, count = target_encoding_map(train_model.iloc[fit_idx], HARD_NEGATIVE_TARGET[fit_idx], apply_frame, cols, CFG.hn_te_alpha)
        out[f'hn_rate__{name}'] = rate; out[f'hn_support__{name}'] = count
    return out.astype(float)

def nested_hn_features(candidate_idx, specs, seed):
    candidate_idx = np.asarray(candidate_idx, dtype=int); target = HARD_NEGATIVE_TARGET[candidate_idx]
    if len(candidate_idx) < CFG.inner_folds or len(np.unique(target)) < 2: raise RuntimeError('empty or single-class specialist candidate pool')
    groups = SIGNATURE_GROUPS.iloc[candidate_idx].to_numpy(); inner = StratifiedGroupKFold(CFG.inner_folds, shuffle=True, random_state=seed)
    train_enc = pd.DataFrame(index=np.arange(len(candidate_idx)))
    for inner_fit, inner_valid in inner.split(np.zeros(len(candidate_idx)), target, groups=groups):
        block = hn_encoding_block(candidate_idx[inner_fit], train_model.iloc[candidate_idx[inner_valid]], specs)
        train_enc.loc[inner_valid, block.columns] = block.to_numpy()
    if train_enc.isna().any().any(): raise RuntimeError('inner target-encoding completeness failed')
    return train_enc.astype(float), hn_encoding_block(candidate_idx, train_model.iloc[candidate_idx], specs)

def specialist_context(frame, idx, base_view, score, rank):
    out = base_view.iloc[idx].reset_index(drop=True).copy()
    out['context__v3_score'] = np.asarray(score)[idx]; out['context__v3_rank'] = np.asarray(rank)[idx]
    out['context__stack_rank'] = STACK_RANK_OOF[idx]
    return out

def fit_hn_model(X_fit, target, X_valid, seed, categorical):
    params = dict(loss_function='Logloss', allow_writing_files=False, verbose=False, random_seed=seed, task_type=CAT_TASK,
                  iterations=CFG.hn_iterations, learning_rate=0.035, depth=7, l2_leaf_reg=10.0, random_strength=0.35,
                  bootstrap_type='Bayesian', bagging_temperature=0.5, max_ctr_complexity=2, border_count=128,
                  class_weights=[1.0, CFG.hn_legitimate_weight])
    if CAT_DEVICES: params['devices'] = CAT_DEVICES
    model = CatBoostClassifier(**params)
    fit_ready = catboost_ready(X_fit, categorical); valid_ready = catboost_ready(X_valid, categorical)
    try: model.fit(fit_ready, target, cat_features=[c for c in categorical if c in fit_ready], verbose=False)
    except Exception as error:
        msg = str(error).lower()
        if CAT_TASK != 'GPU' or not any(t in msg for t in ('gpu', 'cuda', 'nccl', 'memory')): raise
        params.update(devices='0', max_ctr_complexity=1); model = CatBoostClassifier(**params)
        model.fit(fit_ready, target, cat_features=[c for c in categorical if c in fit_ready], verbose=False)
    return model.predict_proba(valid_ready)[:, 1], model

def specialist_primary(view_name):
    base_view = train_model[FULL_FEATURES if view_name == 'full' else PROXY_LIGHT_FEATURES]
    specs = HN_SPECS_FULL if view_name == 'full' else HN_SPECS_PROXY
    candidate_all = V3_CAND_OOF
    oof = np.full(len(y), np.nan)
    categorical = [c for c in base_view if c in CATEGORICAL_COLUMNS]
    for fold, (fit_idx, valid_idx) in enumerate(PRIMARY_SPLITS):
        fit_candidates = fit_idx[candidate_all[fit_idx]]; valid_candidates = valid_idx[candidate_all[valid_idx]]
        train_enc, _ = nested_hn_features(fit_candidates, specs, CFG.seed + 5000 + fold)
        valid_enc = hn_encoding_block(fit_candidates, train_model.iloc[valid_candidates], specs)
        X_fit = specialist_context(train_model[FULL_FEATURES if view_name == 'full' else PROXY_LIGHT_FEATURES], fit_candidates, base_view, V3_OOF_SCORE, V3_FOLD_RANK_OOF)
        X_valid = specialist_context(train_model[FULL_FEATURES if view_name == 'full' else PROXY_LIGHT_FEATURES], valid_candidates, base_view, V3_OOF_SCORE, V3_FOLD_RANK_OOF)
        X_fit = pd.concat([X_fit, train_enc.reset_index(drop=True)], axis=1); X_valid = pd.concat([X_valid, valid_enc.reset_index(drop=True)], axis=1)
        oof[valid_candidates], _ = fit_hn_model(X_fit, HARD_NEGATIVE_TARGET[fit_candidates], X_valid, CFG.seed + 5100 + fold, categorical)
        gc.collect()
    if np.isnan(oof[candidate_all]).any(): raise RuntimeError(f'{view_name} specialist OOF incomplete')
    test_candidate = V3_CAND_TEST; full_candidates = np.flatnonzero(candidate_all)
    train_enc, test_enc = nested_hn_features(full_candidates, specs, CFG.seed + 5500)
    X_fit = specialist_context(train_model[FULL_FEATURES if view_name == 'full' else PROXY_LIGHT_FEATURES], full_candidates, base_view, V3_OOF_SCORE, V3_FOLD_RANK_OOF)
    test_base = test_model[FULL_FEATURES if view_name == 'full' else PROXY_LIGHT_FEATURES].reset_index(drop=True).iloc[np.flatnonzero(test_candidate)]
    X_test = test_base.copy(); X_test['context__v3_score'] = V3_TEST_SCORE[test_candidate]; X_test['context__v3_rank'] = V3_RANK_TEST[test_candidate]; X_test['context__stack_rank'] = STACK_RANK_TEST[test_candidate]
    X_fit = pd.concat([X_fit, train_enc.reset_index(drop=True)], axis=1); X_test = pd.concat([X_test.reset_index(drop=True), hn_encoding_block(full_candidates, test_model.iloc[np.flatnonzero(test_candidate)], specs).reset_index(drop=True)], axis=1)
    test_predictions = []
    for seed_offset, full_seed in enumerate(CFG.specialist_full_seeds):
        pred_test, _ = fit_hn_model(X_fit, HARD_NEGATIVE_TARGET[full_candidates], X_test, int(full_seed) + seed_offset, categorical)
        test_predictions.append(pred_test)
    pred_test = np.mean(test_predictions, axis=0)
    test_oof = np.full(len(test_model), np.nan); test_oof[test_candidate] = pred_test
    return oof, test_oof
if EVAL_FROM_CHECKPOINT:
    HN_FULL_OOF = HN_FULL_TEST = HN_PROXY_OOF = HN_PROXY_TEST = None
    print('Behavior 6 inner and outer target-encoding completeness: inherited from checkpoint evaluation; specialist training skipped.')
else:
    HN_FULL_OOF, HN_FULL_TEST = specialist_primary('full'); HN_PROXY_OOF, HN_PROXY_TEST = specialist_primary('proxy')
    assert np.isfinite(HN_FULL_OOF[V3_CAND_OOF]).all() and np.isfinite(HN_PROXY_OOF[V3_CAND_OOF]).all()
    print('Behavior 6 inner and outer target-encoding completeness: PASS')


## 7. Transport stress and surgical candidate matrix

In [ ]:
def surgical_rerank(base_score, specialist_probability, rank_reference, tie_keys, locked_rate, outer_rate, weight):
    base_score = np.asarray(base_score, dtype=float).copy(); specialist_probability = np.asarray(specialist_probability, dtype=float)
    rank_reference = np.asarray(rank_reference, dtype=float); action = (rank_reference >= 1 - outer_rate - 1e-12) & (rank_reference < 1 - locked_rate - 1e-12)
    idx = np.flatnonzero(action & np.isfinite(specialist_probability))
    if not len(idx): return base_score
    base_local = pd.Series(base_score[idx]).rank(method='average', pct=True).to_numpy()
    specialist_risk = 1.0 - specialist_probability[idx]
    spec_local = pd.Series(specialist_risk).rank(method='average', pct=True).to_numpy()
    mixed = (1 - weight) * base_local + weight * spec_local
    lo, hi = float(base_score[idx].min()), float(base_score[idx].max())
    out = base_score.copy(); out[idx] = lo + (hi - lo) * mixed
    return out

def churn_stats(old_score, new_score, tie_keys):
    old = top_k_mask(old_score, 0.05, tie_keys); new = top_k_mask(new_score, 0.05, tie_keys)
    one_way = int((old & ~new).sum()); return one_way, float(one_way / max(1, old.sum())), float((old & new).sum() / max(1, old.sum()))

def stress_scores(group_col, view_name):
    groups = train_clean[group_col].astype('string').fillna('__MISSING__').to_numpy(); n = min(CFG.stress_folds, pd.Series(groups).nunique())
    splits = list(StratifiedGroupKFold(n, shuffle=True, random_state=CFG.seed + len(group_col)).split(np.zeros(len(y)), y, groups=groups))
    base_view = train_model[FULL_FEATURES if view_name == 'full' else PROXY_LIGHT_FEATURES]; specs = HN_SPECS_FULL if view_name == 'full' else HN_SPECS_PROXY
    candidate_all = V3_CAND_OOF; scores = np.full(len(y), np.nan); categorical = [c for c in base_view if c in CATEGORICAL_COLUMNS]
    for fold, (fit_idx, valid_idx) in enumerate(splits):
        assert not (set(groups[fit_idx]) & set(groups[valid_idx]))
        fit_candidates = fit_idx[candidate_all[fit_idx]]; valid_candidates = valid_idx[candidate_all[valid_idx]]
        train_enc, _ = nested_hn_features(fit_candidates, specs, CFG.seed + 7000 + fold)
        valid_enc = hn_encoding_block(fit_candidates, train_model.iloc[valid_candidates], specs)
        X_fit = specialist_context(base_view, fit_candidates, base_view, V3_OOF_SCORE, V3_FOLD_RANK_OOF)
        X_valid = specialist_context(base_view, valid_candidates, base_view, V3_OOF_SCORE, V3_FOLD_RANK_OOF)
        X_fit = pd.concat([X_fit, train_enc.reset_index(drop=True)], axis=1); X_valid = pd.concat([X_valid, valid_enc.reset_index(drop=True)], axis=1)
        scores[valid_candidates], _ = fit_hn_model(X_fit, HARD_NEGATIVE_TARGET[fit_candidates], X_valid, CFG.seed + 7100 + fold, categorical)
    if np.isnan(scores[candidate_all]).any(): raise RuntimeError(f'{group_col} stress specialist incomplete')
    return scores, splits
CANDIDATES = [{'candidate': 'V3 control', 'family': 'control', 'band': None, 'weight': None},
              {'candidate': 'Stack control', 'family': 'control_stack', 'band': None, 'weight': None}]
for view_name in ['HN full', 'HN proxy-light']:
    for band_idx, (locked, outer) in enumerate(CFG.action_bands, 1):
        for weight in CFG.surgical_weights:
            CANDIDATES.append({'candidate': f'{view_name} band{band_idx} w{weight:.2f}', 'family': view_name, 'band': (locked, outer), 'weight': weight})
assert len(CANDIDATES) == 14 and len({c['candidate'] for c in CANDIDATES}) == len(CANDIDATES)
print(f'Behavior 8 candidate registry predeclared uniqueness: PASS ({len(CANDIDATES)} candidates)')

def score_candidate(candidate, split='oof'):
    if candidate['family'] == 'control': return V3_OOF_SCORE if split == 'oof' else V3_TEST_SCORE
    if candidate['family'] == 'control_stack': return STACK_RANK_OOF if split == 'oof' else STACK_RANK_TEST
    full = candidate['family'] == 'HN full'; specialist = (HN_FULL_OOF if full else HN_PROXY_OOF) if split == 'oof' else (HN_FULL_TEST if full else HN_PROXY_TEST)
    return surgical_rerank(V3_OOF_SCORE if split == 'oof' else V3_TEST_SCORE, specialist,
                           V3_FOLD_RANK_OOF if split == 'oof' else V3_RANK_TEST, TIE_KEYS if split == 'oof' else TEST_TIE_KEYS,
                           *candidate['band'], candidate['weight'])

V3_REBUILD_DELTA = {}
if EVAL_FROM_CHECKPOINT:
    assert set(c['candidate'] for c in CANDIDATES) == set(CKPT_REGISTRY.candidate)
    REGISTRY = CKPT_REGISTRY.copy(); FOLD_DELTAS = CKPT_FOLD_DELTAS.copy(); STRESS_METRICS = CKPT_STRESS_METRICS.copy(); TEST_CHURN = CKPT_TEST_CHURN.copy()
    stored_control = REGISTRY.loc[REGISTRY.candidate.eq('V3 control')].iloc[0]
    nr_delta = float(V3_METRIC['NormalizedRecall@5%'] - stored_control['NormalizedRecall@5%']); tp_delta = int(V3_METRIC['fraud_captured'] - stored_control['fraud_captured'])
    if abs(nr_delta) > 0.001 or abs(tp_delta) > 8:
        raise RuntimeError(f'checkpoint V3 control is inconsistent with the rebuilt control: NR delta {nr_delta:+.6f}, TP delta {tp_delta:+d}')
    if nr_delta or tp_delta:
        print(f'WARNING: rebuilt V3 control differs from checkpoint values: NR delta {nr_delta:+.6f}, TP delta {tp_delta:+d}; rebuilt scores are authoritative for the submission.')
    V3_REBUILD_DELTA = {'normalized_recall_delta': nr_delta, 'tp_delta': tp_delta}
    REGISTRY.to_csv(OUTPUT_DIR / 'v8_candidate_registry.csv', index=False); FOLD_DELTAS.to_csv(OUTPUT_DIR / 'v8_primary_fold_deltas.csv', index=False)
    STRESS_METRICS.to_csv(OUTPUT_DIR / 'v8_stress_metrics.csv', index=False); TEST_CHURN.to_csv(OUTPUT_DIR / 'v8_test_churn.csv', index=False)
    print(f'Checkpoint V3 control matches rebuild within tolerance: NR delta {nr_delta:+.6f}, TP delta {tp_delta:+d}.')
    print('Behavior 3 KDKC/dati2 stress-group disjointness: inherited from checkpoint evaluation.')
    print('Behavior 7 surgical reranking outside-band invariance: inherited from checkpoint evaluation.')
else:
    STRESS = {}
    for group_col in ['kdkc', 'dati2']:
        STRESS[(group_col, 'full')], stress_splits = stress_scores(group_col, 'full')
        STRESS[(group_col, 'proxy')], _ = stress_scores(group_col, 'proxy')
        STRESS[(group_col, 'splits')] = stress_splits
    print('Behavior 3 KDKC/dati2 stress-group disjointness: PASS')

    def stress_metric(candidate, group_col, view_name):
        if candidate['family'] not in {'HN full', 'HN proxy-light'}: return 0.0
        specialist = STRESS[(group_col, 'full' if candidate['family'] == 'HN full' else 'proxy')]
        splits = STRESS[(group_col, 'splits')]; deltas = []
        for _, valid_idx in splits:
            idx = valid_idx
            candidate_score = surgical_rerank(V3_OOF_SCORE[idx], specialist[idx], V3_FOLD_RANK_OOF[idx], TIE_KEYS[idx], *candidate['band'], candidate['weight'])
            deltas.append(normalized_recall_at_capacity(y[idx], candidate_score, 0.05, TIE_KEYS[idx]) - normalized_recall_at_capacity(y[idx], V3_OOF_SCORE[idx], 0.05, TIE_KEYS[idx]))
        return float(np.nanmean(deltas))

    registry_rows = []; fold_delta_rows = []; stress_rows = []; churn_rows = []
    base_nr = V3_METRIC['NormalizedRecall@5%']; base_ap = V3_METRIC['AveragePrecision']; base_fold = fold_metrics(y, V3_OOF_SCORE, FOLD_ID, TIE_KEYS)
    for candidate in CANDIDATES:
        oof_score, test_score = score_candidate(candidate, 'oof'), score_candidate(candidate, 'test')
        metrics = metric_row(y, oof_score, TIE_KEYS, train_clean['jkpst'], train_clean['umur'], candidate['candidate'])
        one_way, oof_churn, overlap = churn_stats(V3_OOF_SCORE, oof_score, TIE_KEYS); test_one_way, test_churn, _ = churn_stats(V3_TEST_SCORE, test_score, TEST_TIE_KEYS)
        folds = fold_metrics(y, oof_score, FOLD_ID, TIE_KEYS).merge(base_fold[['fold', 'tp']], on='fold', suffixes=('', '_v3')); folds['tp_delta'] = folds.tp - folds.tp_v3
        for _, row in folds.iterrows(): fold_delta_rows.append({'candidate': candidate['candidate'], **row.to_dict()})
        confirmation_gain = int(folds.loc[folds.fold > min(3, CFG.folds), 'tp_delta'].sum())
        stress_k = stress_metric(candidate, 'kdkc', candidate['family']); stress_d = stress_metric(candidate, 'dati2', candidate['family'])
        gate_values = [metrics['NormalizedRecall@5%'] - base_nr >= CFG.min_nr5_gain, int((folds.tp_delta > 0).sum()) >= CFG.min_fold_wins,
                       confirmation_gain >= CFG.min_confirmation_gain, float(folds.tp_delta.min()) >= CFG.max_worst_fold_loss,
                       metrics['AveragePrecision'] - base_ap >= CFG.min_ap_delta, oof_churn <= CFG.max_oof_churn,
                       (metrics['fraud_captured'] - V3_METRIC['fraud_captured']) / max(1, one_way) * 100 >= CFG.min_gain_per_100_swaps,
                       test_churn <= min(CFG.max_test_churn, 1.25 * oof_churn + 0.015), stress_k >= 0, stress_d >= 0]
        row = {**metrics, 'family': candidate['family'], 'band': str(candidate['band']), 'weight': candidate['weight'],
               'primary_NR5_delta': metrics['NormalizedRecall@5%'] - base_nr, 'AP_delta': metrics['AveragePrecision'] - base_ap,
               'fold_wins': int((folds.tp_delta > 0).sum()), 'confirmation_tp_gain': confirmation_gain, 'worst_fold_tp_delta': float(folds.tp_delta.min()),
               'one_way_swaps': one_way, 'oof_churn': oof_churn, 'overlap': overlap, 'tp_gain_per_100_swaps': (metrics['fraud_captured'] - V3_METRIC['fraud_captured']) / max(1, one_way) * 100,
               'test_churn': test_churn, 'kdkc_stress_delta': stress_k, 'dati2_stress_delta': stress_d,
               'min_stress_delta': min(stress_k, stress_d)}
        for i, passed in enumerate(gate_values, 1): row[f'gate_{i}'] = bool(passed)
        registry_rows.append(row); stress_rows.append({'candidate': candidate['candidate'], 'kdkc_stress_delta': stress_k, 'dati2_stress_delta': stress_d})
        churn_rows.append({'candidate': candidate['candidate'], 'oof_one_way_swaps': one_way, 'oof_churn': oof_churn, 'test_one_way_swaps': test_one_way, 'test_churn': test_churn})
    REGISTRY = pd.DataFrame(registry_rows); FOLD_DELTAS = pd.DataFrame(fold_delta_rows); STRESS_METRICS = pd.DataFrame(stress_rows); TEST_CHURN = pd.DataFrame(churn_rows)
    REGISTRY.to_csv(OUTPUT_DIR / 'v8_candidate_registry.csv', index=False); FOLD_DELTAS.to_csv(OUTPUT_DIR / 'v8_primary_fold_deltas.csv', index=False)
    print(REGISTRY[['candidate'] + [f'gate_{i}' for i in range(1, 11)]].to_string(index=False))
    STRESS_METRICS.to_csv(OUTPUT_DIR / 'v8_stress_metrics.csv', index=False); TEST_CHURN.to_csv(OUTPUT_DIR / 'v8_test_churn.csv', index=False)
    # Outside the action band, V3 scores are preserved exactly for every challenger.
    for candidate in CANDIDATES[2:]:
        candidate_score = score_candidate(candidate, 'oof'); locked, outer = candidate['band']; action = (V3_FOLD_RANK_OOF >= 1-outer-1e-12) & (V3_FOLD_RANK_OOF < 1-locked-1e-12)
        assert np.array_equal(candidate_score[~action], V3_OOF_SCORE[~action])
    print('Behavior 7 surgical reranking outside-band invariance: PASS')


## 8. Promotion gates, monotone calibration, and submission

In [ ]:
GATE_COLUMNS = ['gate_1', 'gate_2', 'gate_3', 'gate_4', 'gate_5', 'gate_6', 'gate_7', 'gate_8', 'gate_9', 'gate_10']
challenger_rows = REGISTRY[REGISTRY.family.isin(['HN full', 'HN proxy-light'])]
eligible = challenger_rows[challenger_rows[GATE_COLUMNS].all(axis=1)].copy()
complexity = {'HN proxy-light': 0, 'HN full': 1}
if len(eligible):
    eligible = eligible.assign(_complexity=eligible.family.map(complexity))
    selected_name = eligible.sort_values(['min_stress_delta', 'confirmation_tp_gain', 'fraud_captured', 'tp_gain_per_100_swaps', 'test_churn', '_complexity'], ascending=[False, False, False, False, True, True]).iloc[0]['candidate']
    selection_reason = 'eligible challenger selected by predeclared stress, confirmation, gain, efficiency, churn, complexity order'
else:
    selected_name = 'V3 control'; selection_reason = 'no challenger passed all ten promotion gates; V3 control selected'
selected_row = REGISTRY.loc[REGISTRY.candidate.eq(selected_name)].iloc[0]
challenger_failed_gates = {row['candidate']: [gate for gate in GATE_COLUMNS if not bool(row[gate])] for row in challenger_rows.to_dict('records')}
failed_gate_counts = {gate: int((~challenger_rows[gate]).sum()) for gate in GATE_COLUMNS}
assert selected_name in set(REGISTRY.candidate)
if EVAL_FROM_CHECKPOINT and selected_name != 'V3 control':
    raise RuntimeError('checkpoint evaluation selected a challenger but challenger score arrays are not stored; remove the checkpoint input or set DAC_FORCE_FULL=1 and rerun for a full recompute')
selected_score = score_candidate(next(c for c in CANDIDATES if c['candidate'] == selected_name), 'oof')
selected_test_score = score_candidate(next(c for c in CANDIDATES if c['candidate'] == selected_name), 'test')
# Determinism check, including the no-eligible fallback behavior.
def select_candidate(registry):
    eligible = registry[(registry.family.isin(['HN full', 'HN proxy-light'])) & registry[GATE_COLUMNS].all(axis=1)]
    if eligible.empty: return 'V3 control'
    eligible = eligible.assign(_complexity=eligible.family.map(complexity))
    return eligible.sort_values(['min_stress_delta', 'confirmation_tp_gain', 'fraud_captured', 'tp_gain_per_100_swaps', 'test_churn', '_complexity'], ascending=[False, False, False, False, True, True]).iloc[0].candidate
assert select_candidate(REGISTRY) == select_candidate(REGISTRY.copy())
assert select_candidate(REGISTRY.assign(**{c: False for c in GATE_COLUMNS})) == 'V3 control'
print('Behavior 9 deterministic promotion and V3 fallback: PASS')
if selected_name == 'V3 control':
    print('No eligible challenger. Challenger failed-gate counts:', failed_gate_counts)
    print('Candidate-specific failed gates:', challenger_failed_gates)

# Fit calibration only after ranking selection. A sigmoid is monotone and must not alter ordering.
calibrator = LogisticRegression(C=1.0, max_iter=CFG.logistic_max_iter, solver='lbfgs', random_state=CFG.seed)
calibrator.fit(probability_logit(np.clip(selected_score, 1e-6, 1-1e-6)).reshape(-1, 1), y)
final_probability = np.clip(calibrator.predict_proba(probability_logit(np.clip(selected_test_score, 1e-6, 1-1e-6)).reshape(-1, 1))[:, 1], 0, 1)
assert np.array_equal(np.argsort(-selected_test_score, kind='mergesort'), np.argsort(-final_probability, kind='mergesort'))

submission = pd.DataFrame({ID_COL: sample_submission[ID_COL].astype(str), 'fraud_probability': final_probability})
assert submission[ID_COL].tolist() == sample_submission[ID_COL].astype(str).tolist()
assert not submission[ID_COL].duplicated().any() and np.isfinite(submission.fraud_probability).all() and submission.fraud_probability.between(0, 1).all()
submission.to_csv(SUBMISSION_PATH, index=False); submission.to_csv(OUTPUT_DIR / 'submission_v8.csv', index=False)
np.savez_compressed(OUTPUT_DIR / 'v8_scores.npz', selected_oof=selected_score.astype(np.float32), selected_test=selected_test_score.astype(np.float32), calibrated_test=final_probability.astype(np.float32), v3_oof=V3_OOF_SCORE.astype(np.float32), v3_test=V3_TEST_SCORE.astype(np.float32))
selected_config = {'selected_candidate': selected_name, 'selection_reason': selection_reason, 'mode': RUNTIME_LABEL, 'official_data_only': True,
                   'config': asdict(CFG), 'environment': {'python': platform.python_version(), 'platform': platform.platform(), 'gpus': GPU_COUNT, 'catboost_devices': CAT_DEVICES or 'CPU', 'hardware': HARDWARE, 'lightgbm_available': LIGHTGBM_AVAILABLE},
                   'primary_metrics': selected_row.to_dict(), 'base_roster': list(BASE_RESULTS), 'fast_mode_roster': {'base_models': list(BASE_RESULTS), 'specialist_seeds': list(CFG.specialist_full_seeds), 'folds': CFG.folds, 'stress_folds': CFG.stress_folds, 'non_comparable': bool(CFG.fast_mode)},
                   'promotion_gates': {gate: bool(selected_row[gate]) for gate in GATE_COLUMNS}, 'challenger_failed_gate_counts': failed_gate_counts, 'challenger_failed_gates': challenger_failed_gates,
                   'artifact_provenance': ARTIFACT_PROVENANCE, 'artifact_fold_assignment_matches_current_primary': ARTIFACT_FOLD_MATCHES_CURRENT, 'claim_id_predictive_feature': False, 'row_position_predictive_feature': False,
                   'resumed_from_v8_checkpoint': bool(EVAL_FROM_CHECKPOINT), 'checkpoint_provenance': CHECKPOINT_PROVENANCE, 'v3_control_rebuild_delta': V3_REBUILD_DELTA,
                   'calibration': 'post-ranking monotone sigmoid', 'candidate_count': len(CANDIDATES), 'non_comparable_fast_mode': bool(CFG.fast_mode)}
with open(OUTPUT_DIR / 'selected_config.json', 'w') as handle: json.dump(selected_config, handle, indent=2, default=str)
manifest = {'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'mode': RUNTIME_LABEL, 'files': sorted(p.name for p in OUTPUT_DIR.iterdir()),
            'artifact_provenance': ARTIFACT_PROVENANCE, 'official_data_only': True}
with open(OUTPUT_DIR / 'artifact_manifest.json', 'w') as handle: json.dump(manifest, handle, indent=2)

# Submission contract is checked as the final behavioral requirement.
assert list(pd.read_csv(OUTPUT_DIR / 'submission_v8.csv').columns) == [ID_COL, 'fraud_probability']
assert len(pd.read_csv(OUTPUT_DIR / 'submission_v8.csv')) == len(sample_submission)
print('Behavior 10 submission schema/order/uniqueness/range/finiteness: PASS')
print(json.dumps({'selected_candidate': selected_name, 'mode': RUNTIME_LABEL, 'submission': str(SUBMISSION_PATH),
                  'output_dir': str(OUTPUT_DIR), 'registry_rows': len(REGISTRY)}, indent=2))
